# Import libraries

In [15]:
from datasets import load_dataset
import pandas as pd
from sklearn_crfsuite import CRF
from sklearn_crfsuite.metrics import flat_classification_report
from sklearn.model_selection import train_test_split
import joblib

In [8]:

dataset = load_dataset("jjzha/skillspan")
print(dataset.keys())

dict_keys(['train', 'validation', 'test'])


In [9]:
test_data = dataset['test']
train_data = dataset['train']
validation_data = dataset['validation']


In [10]:
df_train = train_data.to_pandas()
df_validation = validation_data.to_pandas()
df_test = test_data.to_pandas()

In [13]:
df_train = pd.concat([df_train, df_validation], ignore_index=True)

In [14]:
df_train

,idx,tokens,tags_skill,tags_knowledge,source
0,1,"[Senior, QA, Engineer, (, m/f/d, ), <ORGANIZAT...","[O, O, O, O, O, O, O]","[O, O, O, O, O, O, O]",tech
1,1,"[<ADDRESS>, <ADDRESS>, <ADDRESS>, <ADDRESS>, <...","[O, O, O, O, O]","[O, O, O, O, O]",tech
2,1,"[Date, posted:, 2021-07-14]","[O, O, O]","[O, O, O]",tech
3,1,"[Likes:, 0, Dislikes:, 0, Love:, 0]","[O, O, O, O, O, O]","[O, O, O, O, O, O]",tech
4,1,"[Job, description:]","[O, O]","[O, O]",tech
...,...,...,...,...,...
7969,58,[Conditions],[O],[O],house
7970,58,"[The, two, positions, are, full-time, position...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]",house
7971,58,"[Salary, and, employment, conditions, are, in,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",house
7972,58,"[We, want, to, reflect, the, surrounding, soci...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",house


In [22]:
import pandas as pd
from sklearn_crfsuite import CRF
from sklearn.model_selection import train_test_split
import joblib
from seqeval.metrics import classification_report, f1_score
import numpy as np

# Load your data
# Assuming you have a DataFrame with 'tokens' and 'tags_skill' columns
# df = pd.read_csv('your_data.csv')  # Load your data here

# Prepare the data for CRF
def word2features(sent, i):
    """
    Extract features for a given word in a sentence
    """
    word = sent[i]
    
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word[-2:]': word[-2:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
        'word.length': len(word),
    }
    
    # Context features (previous and next words)
    if i > 0:
        prev_word = sent[i-1]
        features.update({
            '-1:word.lower()': prev_word.lower(),
            '-1:word.istitle()': prev_word.istitle(),
            '-1:word.isupper()': prev_word.isupper(),
        })
    else:
        features['BOS'] = True  # Beginning of sentence
        
    if i < len(sent)-1:
        next_word = sent[i+1]
        features.update({
            '+1:word.lower()': next_word.lower(),
            '+1:word.istitle()': next_word.istitle(),
            '+1:word.isupper()': next_word.isupper(),
        })
    else:
        features['EOS'] = True  # End of sentence
        
    return features

def sent2features(sent):
    """
    Convert a sentence to features
    """
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    """
    Convert a sentence to labels
    """
    return sent

# Prepare the features and labels
X = [sent2features(s) for s in df_train['tokens']]
y = df_train['tags_skill'].tolist()

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train CRF model
crf = CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)

crf.fit(X_train, y_train)

# Evaluate on test set
y_pred = crf.predict(X_test)

# Ensure the labels are in the correct format for seqeval
# Convert any numpy arrays to lists of strings
y_test_formatted = []
for seq in y_test:
    if hasattr(seq, 'tolist'):
        y_test_formatted.append([str(label) for label in seq.tolist()])
    else:
        y_test_formatted.append([str(label) for label in seq])

y_pred_formatted = []
for seq in y_pred:
    if hasattr(seq, 'tolist'):
        y_pred_formatted.append([str(label) for label in seq.tolist()])
    else:
        y_pred_formatted.append([str(label) for label in seq])

# Calculate span-level F1 score
print("CRF Performance:")
print(classification_report(y_test_formatted, y_pred_formatted))
f1 = f1_score(y_test_formatted, y_pred_formatted)
print(f"Span-level F1: {f1:.4f}")

# Save the model
joblib.dump(crf, 'crf_model.joblib')
print("CRF model saved as 'crf_model.joblib'")

# Check if F1 score meets the requirement
if f1 >= 0.6:
    print("✅ CRF model achieved F1 ≥ 60%")
else:
    print("❌ CRF model did not achieve F1 ≥ 60%")
    print("Consider feature engineering or hyperparameter tuning")

CRF Performance:
              precision    recall  f1-score   support

           _       0.43      0.24      0.31       755

   micro avg       0.43      0.24      0.31       755
   macro avg       0.43      0.24      0.31       755
weighted avg       0.43      0.24      0.31       755

Span-level F1: 0.3056
CRF model saved as 'crf_model.joblib'
❌ CRF model did not achieve F1 ≥ 60%
Consider feature engineering or hyperparameter tuning


In [23]:
import pandas as pd
from sklearn_crfsuite import CRF
from sklearn.model_selection import train_test_split
import joblib
from seqeval.metrics import classification_report, f1_score
import numpy as np
from collections import Counter

# First, let's examine the tag distribution in your data
print("Tag distribution in training data:")
all_tags = [tag for sublist in df_train['tags_skill'] for tag in sublist]
tag_counts = Counter(all_tags)
print(tag_counts)

# Check if tags are in BIO format
print("\nUnique tags:")
print(set(all_tags))

# Let's also check a few examples to understand the data format
print("\nSample tokens and tags:")
for i in range(min(3, len(df_train))):
    print(f"Example {i}:")
    print("Tokens:", df_train['tokens'].iloc[i])
    print("Tags:", df_train['tags_skill'].iloc[i])
    print()

Tag distribution in training data:
Counter({'O': 118572, 'I': 11647, 'B': 3291})

Unique tags:
{'B', 'O', 'I'}

Sample tokens and tags:
Example 0:
Tokens: ['Senior' 'QA' 'Engineer' '(' 'm/f/d' ')' '<ORGANIZATION>']
Tags: ['O' 'O' 'O' 'O' 'O' 'O' 'O']

Example 1:
Tokens: ['<ADDRESS>' '<ADDRESS>' '<ADDRESS>' '<ADDRESS>' '<LOCATION>']
Tags: ['O' 'O' 'O' 'O' 'O']

Example 2:
Tokens: ['Date' 'posted:' '2021-07-14']
Tags: ['O' 'O' 'O']



In [24]:
# Improved CRF implementation with better features
def word2features(sent, i):
    """
    Extract enhanced features for a given word in a sentence
    """
    word = sent[i]
    
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word[-2:]': word[-2:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
        'word.length': len(word),
        'word.contains_hyphen': '-' in word,
        'word.contains_digit': any(char.isdigit() for char in word),
        'word.contains_uppercase': any(char.isupper() for char in word),
    }
    
    # Add context features (previous and next words)
    for offset in range(1, 3):  # Look at previous 2 and next 2 words
        if i - offset >= 0:
            prev_word = sent[i-offset]
            features.update({
                f'-{offset}:word.lower()': prev_word.lower(),
                f'-{offset}:word.istitle()': prev_word.istitle(),
                f'-{offset}:word.isupper()': prev_word.isupper(),
                f'-{offset}:word.isdigit()': prev_word.isdigit(),
            })
        else:
            features[f'-{offset}:BOS'] = True  # Beginning of sentence
            
        if i + offset < len(sent):
            next_word = sent[i+offset]
            features.update({
                f'+{offset}:word.lower()': next_word.lower(),
                f'+{offset}:word.istitle()': next_word.istitle(),
                f'+{offset}:word.isupper()': next_word.isupper(),
                f'+{offset}:word.isdigit()': next_word.isdigit(),
            })
        else:
            features[f'+{offset}:EOS'] = True  # End of sentence
    
    # Add position features
    features['position_in_sentence'] = i / len(sent)
    
    return features

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return sent

# Prepare the features and labels
X = [sent2features(s) for s in df_train['tokens']]
y = df_train['tags_skill'].tolist()

# Split the data into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Try different hyperparameters
crf = CRF(
    algorithm='lbfgs',
    c1=0.01,  # Reduced regularization
    c2=0.01,  # Reduced regularization
    max_iterations=200,  # Increased iterations
    all_possible_transitions=True,
    all_possible_states=True
)

# Train the model
crf.fit(X_train, y_train)

# Evaluate on validation set
y_pred = crf.predict(X_val)

# Format for seqeval
y_val_formatted = [[str(tag) for tag in seq] for seq in y_val]
y_pred_formatted = [[str(tag) for tag in seq] for seq in y_pred]

# Calculate span-level F1 score
print("Improved CRF Performance:")
print(classification_report(y_val_formatted, y_pred_formatted))
f1 = f1_score(y_val_formatted, y_pred_formatted)
print(f"Span-level F1: {f1:.4f}")

# Save the model
joblib.dump(crf, 'crf_model_improved.joblib')
print("Improved CRF model saved as 'crf_model_improved.joblib'")

# Check if F1 score meets the requirement
if f1 >= 0.6:
    print("✅ CRF model achieved F1 ≥ 60%")
else:
    print("❌ CRF model did not achieve F1 ≥ 60%")
    
    # If still not meeting requirements, let's try a different approach
    print("\nTrying alternative approach with different algorithm...")
    
    # Try with different algorithm
    crf2 = CRF(
        algorithm='ap',
        max_iterations=100,
        all_possible_transitions=True
    )
    
    crf2.fit(X_train, y_train)
    y_pred2 = crf2.predict(X_val)
    y_pred2_formatted = [[str(tag) for tag in seq] for seq in y_pred2]
    
    f1_2 = f1_score(y_val_formatted, y_pred2_formatted)
    print(f"Alternative algorithm F1: {f1_2:.4f}")
    
    if f1_2 > f1:
        joblib.dump(crf2, 'crf_model_alternative.joblib')
        print("Alternative CRF model saved as 'crf_model_alternative.joblib'")

Improved CRF Performance:
              precision    recall  f1-score   support

           _       0.34      0.26      0.30       755

   micro avg       0.34      0.26      0.30       755
   macro avg       0.34      0.26      0.30       755
weighted avg       0.34      0.26      0.30       755

Span-level F1: 0.2958
Improved CRF model saved as 'crf_model_improved.joblib'
❌ CRF model did not achieve F1 ≥ 60%

Trying alternative approach with different algorithm...
Alternative algorithm F1: 0.1647
